In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

print(train.shape)
train.head()

In [ ]:
train.info()
train.describe()
train.isnull().sum()

In [ ]:
sns.countplot(data=train, x='Sex', hue='Survived')
plt.title('Survival by Sex')
plt.show()
sns.countplot(data=train, x='Pclass', hue='Survived')
plt.title('Survival by Class')
plt.show()

In [ ]:
# Imputation
train['Age'].fillna(train['Age'].median(), inplace=True)
test['Age'].fillna(train['Age'].median(), inplace=True)
train['Embarked'].fillna(train['Embarked'].mode()[0], inplace=True)
test['Fare'].fillna(test['Fare'].median(), inplace=True)

# Encoding
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})
train['Embarked'] = train['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
test['Embarked'] = test['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

In [ ]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

X_train = train[features]
y_train = train['Survived']
X_test = test[features]

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Step 1: Baseline linear kernel
linear_model = SVC(kernel='linear', random_state=42)
linear_model.fit(X_train_scaled, y_train)
linear_preds = linear_model.predict(X_train_scaled)
print('Linear Kernel Training Accuracy:', accuracy_score(y_train, linear_preds))

In [ ]:
# Step 2: RBF with C=1, 5, 10
for C_val in [1, 5, 10]:
    rbf_model = SVC(kernel='rbf', C=C_val, gamma='scale', random_state=42)
    rbf_model.fit(X_train_scaled, y_train)
    preds = rbf_model.predict(X_train_scaled)
    print(f'RBF Kernel (C={C_val}) Training Accuracy: {accuracy_score(y_train, preds):.4f}')

In [ ]:
# Step 3: GridSearchCV over C, gamma, and kernel
param_grid = {
    'kernel': ['rbf', 'linear'],
    'C':      [0.1, 1, 5, 10, 50],
    'gamma':  ['scale', 'auto', 0.01, 0.001]
}

grid_search = GridSearchCV(
    SVC(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_scaled, y_train)

print('\nBest Parameters :', grid_search.best_params_)
print('Best CV Accuracy :', round(grid_search.best_score_, 4))

In [ ]:
# Step 4: Final model with best params
best_model = grid_search.best_estimator_

train_preds = best_model.predict(X_train_scaled)
print('Final Training Accuracy:', round(accuracy_score(y_train, train_preds), 4))

In [ ]:
# Step 5: Submission
predictions = best_model.predict(X_test_scaled)
submission = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': predictions})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print('\nDone! File saved.')
submission.head(10)